# Lab: four models, one split

The risk team reviews clients who might default next month. One row is one credit-card client; `default` is 1 when the client defaulted. The team asks: "Which model ranks the risky clients best, and how many defaulters would it find if we review the top 10%?"

Work through the checkpoints in order. C1 is the minimum, C2 is today's target, C3 is for those who finish C2.

## The starter

The cell loads the data and splits it 80 / 20 with `stratify=y, random_state=42`, the same split as 5.2. `scores(model)` takes a fitted model and returns accuracy, ROC AUC, average precision, and precision, recall and F1 at the default 0.5 threshold, all on the test rows.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split

NAMES = (["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
         + [f"BILL_AMT{i}" for i in range(1, 7)] + [f"PAY_AMT{i}" for i in range(1, 7)])

df = pd.read_csv("data/credit_default.csv")
df = df.drop(columns=[c for c in df.columns if c.upper() == "ID"])
target = [c for c in df.columns if c.lower() in ("y", "default", "default.payment.next.month",
                                                 "default payment next month")][0]
y = df.pop(target).astype(int).rename("default")
if list(df.columns[:2]) == ["X1", "X2"]:            # the UCI download names the columns X1 ... X23
    df.columns = NAMES
X = df.rename(columns={"PAY_1": "PAY_0"})[NAMES]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)


def scores(model):
    """Test-row metrics for a fitted classifier; precision, recall and F1 at the 0.5 threshold."""
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    return {"accuracy": accuracy_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba),
            "avg_precision": average_precision_score(y_test, proba),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred, zero_division=0)}


print(len(X_train), "training clients,", len(X_test), "test clients,", y_test.sum(), "test defaulters",
      f"(base rate {y_test.mean():.3f})")

24000 training clients, 6000 test clients, 1327 test defaulters (base rate 0.221)


## C1 · Two baselines

### Step 1 · Score a `DummyClassifier(strategy="most_frequent")`

In [2]:
from sklearn.dummy import DummyClassifier

results = {}
results["Dummy"] = scores(DummyClassifier(strategy="most_frequent").fit(X_train, y_train))
results["Dummy"]

{'accuracy': 0.7788333333333334,
 'roc_auc': 0.5,
 'avg_precision': 0.22116666666666668,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0}

### Step 2 · Fit `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` and score it

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
results["Logistic"] = scores(logistic)

### Step 3 · Fit `DecisionTreeClassifier(max_depth=5, random_state=42)` and score it

In [4]:
from sklearn.tree import DecisionTreeClassifier

results["Tree, depth 5"] = scores(DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train))

## C2 · Two ensembles

### Step 4 · Fit `RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)` and score it

It is the slowest of the four to fit.

In [5]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
results["Random forest"] = scores(forest)

### Step 5 · Fit `HistGradientBoostingClassifier(random_state=42)` and score it

Print `n_iter_`: early stopping is on by default when there are more than 10,000 rows.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosted = HistGradientBoostingClassifier(random_state=42).fit(X_train, y_train)
results["Boosted"] = scores(boosted)
print("n_iter_", boosted.n_iter_, "of a maximum of", boosted.max_iter)

n_iter_ 72 of a maximum of 100


### Step 6 · One table, sorted by ROC AUC

In [7]:
table = pd.DataFrame(results).T.sort_values("roc_auc", ascending=False).round(3)
table

,accuracy,roc_auc,avg_precision,precision,recall,f1
Boosted,0.817,0.781,0.556,0.656,0.363,0.468
Random forest,0.814,0.755,0.536,0.637,0.365,0.464
"Tree, depth 5",0.817,0.742,0.488,0.661,0.356,0.463
Logistic,0.808,0.708,0.494,0.687,0.240,0.355
Dummy,0.779,0.500,0.221,0.000,0.000,0.000


### Step 7 · "Which model ranks the risky clients best?"

Translate: the highest ROC AUC and average precision. Then say why the accuracy column could not have answered the question.

In [8]:
models = table.drop(index="Dummy")
print(f"{models.index[0]} ranks best: ROC AUC {models['roc_auc'].iloc[0]:.3f}, average precision "
      f"{models.loc[models.index[0], 'avg_precision']:.3f}. Accuracy spans only {models['accuracy'].min():.3f} to "
      f"{models['accuracy'].max():.3f}, because a model that says 'no default' to everyone already scores "
      f"{table.loc['Dummy', 'accuracy']:.3f}; ROC AUC spans {models['roc_auc'].min():.3f} to {models['roc_auc'].max():.3f}.")

Boosted ranks best: ROC AUC 0.781, average precision 0.556. Accuracy spans only 0.808 to 0.817, because a model that says 'no default' to everyone already scores 0.779; ROC AUC spans 0.708 to 0.781.


## C3 · The review list

### Step 8 · The 600 clients each model ranks highest

For the boosted model and the logistic regression, take the 600 test clients with the highest predicted probability (10% of 6,000) and count how many defaulted.

In [9]:
top600 = {}
for name, m in (("Boosted", boosted), ("Logistic", logistic)):
    top = np.argsort(-m.predict_proba(X_test)[:, 1])[:600]
    top600[name] = y_test.values[top].sum()
    print(f"{name}: {top600[name]} defaulters in the top 600 ({top600[name] / 600:.3f})")
print("expected at random:", round(600 * y_test.mean()))

Boosted: 419 defaulters in the top 600 (0.698)
Logistic: 401 defaulters in the top 600 (0.668)
expected at random: 133


### Step 9 · `class_weight="balanced"`

Refit both with `class_weight="balanced"`. Print precision, recall and ROC AUC again. What changed, and what did not?

In [10]:
logistic_bal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_train, y_train)
boosted_bal = HistGradientBoostingClassifier(class_weight="balanced", random_state=42).fit(X_train, y_train)
balanced = {"Logistic, balanced": scores(logistic_bal), "Boosted, balanced": scores(boosted_bal)}
pd.DataFrame(balanced).T[["precision", "recall", "roc_auc"]].round(3)

,precision,recall,roc_auc
"Logistic, balanced",0.367,0.620,0.708
"Boosted, balanced",0.477,0.623,0.776


### Step 10 · Three sentences for the risk team

The model you recommend, its test ROC AUC, and how many defaulters it puts in a review list of 600.

In [11]:
print(f"We recommend the boosted model: it ranks risky clients best, with a test ROC AUC of "
      f"{results['Boosted']['roc_auc']:.3f}. A review list of its top 600 clients holds {top600['Boosted']} "
      f"defaulters, against about {round(600 * y_test.mean())} for a list picked at random. Judge it by that "
      f"list, not by accuracy, which barely separates the models.")

We recommend the boosted model: it ranks risky clients best, with a test ROC AUC of 0.781. A review list of its top 600 clients holds 419 defaulters, against about 133 for a list picked at random. Judge it by that list, not by accuracy, which barely separates the models.


## Bonus · A forest of shallower trees

In [12]:
forest8 = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(X_train, y_train)
print("max_depth=8: test AUC", round(scores(forest8)["roc_auc"], 3), "against", round(results["Random forest"]["roc_auc"], 3))

max_depth=8: test AUC 0.773 against 0.755


## Check yourself

In [13]:
check = {"Step 1 · training clients": len(X_train), "Step 1 · test clients": len(X_test),
         "Step 1 · test defaulters": y_test.sum(), "Step 1 · base rate": round(y_test.mean(), 3)}
for name, key in (("Dummy", "dummy"), ("Logistic", "logistic"), ("Tree, depth 5", "tree"),
                  ("Random forest", "forest"), ("Boosted", "boosted")):
    for metric in ("accuracy", "roc_auc", "avg_precision", "precision", "recall", "f1"):
        check[f"Table · {key} {metric}"] = round(results[name][metric], 3)
check["Step 5 · n_iter_"] = boosted.n_iter_
check["Step 5 · max_iter"] = boosted.max_iter
check["Step 8 · boosted top 600"] = top600["Boosted"]
check["Step 8 · logistic top 600"] = top600["Logistic"]
check["Step 8 · expected at random"] = round(600 * y_test.mean())
for name, key in (("Logistic, balanced", "logistic_bal"), ("Boosted, balanced", "boosted_bal")):
    for metric in ("precision", "recall", "roc_auc"):
        check[f"Step 9 · {key} {metric}"] = round(balanced[name][metric], 3)
check["Bonus · forest max_depth=8 AUC"] = round(scores(forest8)["roc_auc"], 3)
for label, value in check.items():
    print(f"{label:<36} {value}")

Step 1 · training clients            24000
Step 1 · test clients                6000
Step 1 · test defaulters             1327
Step 1 · base rate                   0.221
Table · dummy accuracy               0.779
Table · dummy roc_auc                0.5
Table · dummy avg_precision          0.221
Table · dummy precision              0.0
Table · dummy recall                 0.0
Table · dummy f1                     0.0
Table · logistic accuracy            0.808
Table · logistic roc_auc             0.708
Table · logistic avg_precision       0.494
Table · logistic precision           0.687
Table · logistic recall              0.24
Table · logistic f1                  0.355
Table · tree accuracy                0.817
Table · tree roc_auc                 0.742
Table · tree avg_precision           0.488
Table · tree precision               0.661
Table · tree recall                  0.356
Table · tree f1                      0.463
Table · forest accuracy              0.814
Table · forest roc_auc